# Trace Analysis Starter

This notebook demonstrates the basic workflow for trace analysis and error coding.

## Setup

Make sure you've:
1. Created a virtual environment: `python3 -m venv venv`
2. Activated it: `source venv/bin/activate`
3. Installed dependencies: `pip install -r requirements.txt`
4. Have DATABASE_URL in your `.env.local`

In [ ]:
# Cell 1: Setup
import sys
sys.path.append('..')  # Add parent directory to path

from scripts.trace_analysis import TraceAnalyzer
import pandas as pd

# Connect to database
analyzer = TraceAnalyzer()

In [ ]:
# Cell 2: Load traces
traces = analyzer.load_traces(limit=50)
print(f"Loaded {len(traces)} traces")

# Quick overview
traces[['id', 'selectedLens', 'questionCount', 'userFeedback', 'reviewedAt']].head()

In [ ]:
# Cell 3: Filter traces (example)
# Example: Find all lens A conversations with negative feedback
problematic = traces[
    (traces['selectedLens'] == 'A') &
    (traces['userFeedback'] == 'not_helpful')
]
print(f"Found {len(problematic)} problematic traces")
problematic[['id', 'questionCount', 'timestamp']]

In [ ]:
# Cell 4: Review individual trace
# Pick first trace from filtered set (or use any trace ID)
if len(traces) > 0:
    trace_id = traces.iloc[0]['id']
    analyzer.display_trace(trace_id)
else:
    print("No traces available. Run the app and create some conversations first.")

In [ ]:
# Cell 5: Add open coding notes
# Record observations as you review
# analyzer.annotate_trace(
#     trace_id=trace_id,
#     notes="""
#     Observation: User provided specific customer pain point,
#     but extraction generalized too much.
#     
#     Potential error: over-generalization in extraction phase
#     Impact: Strategy output was too vague
#     """
# )

In [ ]:
# Cell 6: View all coded traces
coded = analyzer.load_traces(limit=100, filters={'hasNotes': True})
print(f"{len(coded)} traces have been coded")

# Browse notes
for idx, row in coded.iterrows():
    print(f"\n--- Trace {row['id'][:8]}... ---")
    print(f"Lens: {row['selectedLens']}, Feedback: {row['userFeedback']}")
    notes = row['openCodingNotes']
    if notes:
        print(notes[:200] + "..." if len(notes) > 200 else notes)

In [ ]:
# Cell 7: Apply categories (after synthesis)
# Once you've identified themes, apply categories
# Example: Categorize traces with extraction issues

# analyzer.categorize_trace(
#     trace_id=trace_id,
#     categories=['extraction-overgeneralization', 'weak-strategy-specificity']
# )

In [ ]:
# Cell 8: Summary statistics
summary = analyzer.get_coding_summary()

print(f"Total traces: {summary['total_traces']}")
print(f"Coded: {summary['coded_traces']} ({summary['coded_traces']/summary['total_traces']*100:.1f}%)")
print(f"Categorized: {summary['categorized_traces']}")
print("\nCategory distribution:")
print(summary['category_distribution'])